In [1]:
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_classic.vectorstores import FAISS


In [2]:
# local embedding model import
MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)


/home/mehedi-hassan/rag-learning-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10683.35it/s]


In [3]:
documents = [
    Document(page_content="LangChain is a framework for LLM apps."),
    Document(page_content="Retrievers find relevant documents."),
    Document(page_content="FAISS performs fast vector search."),
    Document(page_content="Chroma is a vector database."),
]

# FAISS vector store তৈরি করছি. from_documents() document গুলো embed করে FAISS index-এ রাখে 
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

# FAISS index local disk-এ save করছি
vector_store.save_local("../all_vector_stores/faiss_index")

In [ ]:

# পরে আবার load করার জন্য:
# vector_store = FAISS.load_local(
#     "../all_vector_stores/faiss_index",
#     embeddings,
#     allow_dangerous_deserialization=True
# )

# FAISS-কে retriever-এ convert করছি
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

query = "What is FAISS?"

results = retriever.invoke(query)

print("Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Query: What is FAISS?
--------------------------------------------------
1. FAISS performs fast vector search.
2. LangChain is a framework for LLM apps.
